In [13]:
import sys
!{sys.executable} -m pip install xgboost

In [14]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb

In [15]:
df=pd.read_csv('data.csv')

In [16]:
df.shape

(103086, 30)

In [17]:
df.head()

,property_type,cancelled,days_before_arrival,arrival_year,arrival_month,arrival_week,arrival_day,weekend_nights,weekday_nights,n_adults,...,room_given,n_changes,deposit,agent_id,company_id,waitlist_days,customer_class,avg_daily_rate,parking_spaces,special_requests
0,Resort Hotel,1,115,2015,October,40,2,2,2,2,...,G,0,No Deposit,240.0,NaN,0,Transient,106.00,0,0
1,Resort Hotel,0,4,2016,September,40,27,0,1,1,...,D,0,No Deposit,NaN,9.0,0,Transient,59.00,0,0
2,City Hotel,0,90,2016,September,37,7,0,3,2,...,D,0,No Deposit,83.0,NaN,0,Transient,112.88,0,0
3,City Hotel,1,38,2017,January,2,14,0,1,1,...,A,0,Non Refund,NaN,67.0,0,Transient,75.00,0,0
4,City Hotel,0,1,2017,July,30,29,4,11,2,...,A,2,No Deposit,14.0,NaN,0,Transient,31.00,0,0


In [18]:
df.tail()

,property_type,cancelled,days_before_arrival,arrival_year,arrival_month,arrival_week,arrival_day,weekend_nights,weekday_nights,n_adults,...,room_given,n_changes,deposit,agent_id,company_id,waitlist_days,customer_class,avg_daily_rate,parking_spaces,special_requests
103081,Resort Hotel,1,92,2016,August,33,12,2,2,2,...,E,0,No Deposit,250.0,NaN,0,Transient,251.50,0,0
103082,City Hotel,1,14,2017,February,9,26,2,2,2,...,A,0,No Deposit,9.0,NaN,0,Transient,108.80,0,0
103083,City Hotel,0,116,2016,May,20,14,2,1,2,...,A,0,No Deposit,9.0,NaN,0,Transient,112.50,0,0
103084,Resort Hotel,1,128,2015,August,33,12,2,8,1,...,D,0,No Deposit,6.0,NaN,0,Transient-Party,90.95,0,1
103085,Resort Hotel,0,17,2015,December,49,3,0,1,2,...,E,0,No Deposit,240.0,NaN,0,Transient,61.20,0,1


In [19]:
df.describe()

,cancelled,days_before_arrival,arrival_year,arrival_week,arrival_day,weekend_nights,weekday_nights,n_adults,n_children,n_babies,returning_guest,past_cancellations,past_completed_stays,n_changes,agent_id,company_id,waitlist_days,avg_daily_rate,parking_spaces,special_requests
count,103086.000000,103086.000000,103086.000000,103086.000000,103086.000000,103086.000000,103086.000000,103086.000000,103083.000000,103086.000000,103086.000000,103086.000000,103086.000000,103086.000000,88787.000000,5909.000000,103086.000000,103086.000000,103086.000000,103086.000000
mean,0.368847,107.235260,2016.146354,27.188241,15.779660,0.912985,2.472596,1.852453,0.097300,0.007654,0.031566,0.095706,0.135334,0.216111,85.503508,191.320697,2.591623,100.904922,0.061327,0.553257
std,0.482495,109.343318,0.710272,13.595908,8.762014,0.992338,1.882054,0.566650,0.386744,0.097251,0.174842,0.899746,1.510703,0.643898,110.223555,132.133168,18.525930,50.329254,0.243543,0.786393
min,0.000000,0.000000,2015.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,6.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,19.000000,2016.000000,16.000000,8.000000,0.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.000000,67.000000,0.000000,68.000000,0.000000,0.000000
50%,0.000000,72.000000,2016.000000,28.000000,16.000000,1.000000,2.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,14.000000,186.000000,0.000000,93.600000,0.000000,0.000000
75%,1.000000,165.000000,2017.000000,38.000000,23.000000,2.000000,3.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,208.000000,273.000000,0.000000,124.200000,0.000000,1.000000
max,1.000000,737.000000,2017.000000,53.000000,31.000000,19.000000,50.000000,55.000000,10.000000,10.000000,1.000000,26.000000,72.000000,21.000000,535.000000,543.000000,391.000000,5400.000000,8.000000,5.000000


In [20]:
df.isnull().sum()

property_type               0
cancelled                   0
days_before_arrival         0
arrival_year                0
arrival_month               0
arrival_week                0
arrival_day                 0
weekend_nights              0
weekday_nights              0
n_adults                    0
n_children                  3
n_babies                    0
meal_plan                   0
origin_code               416
segment                     0
channel                     0
returning_guest             0
past_cancellations          0
past_completed_stays        0
room_booked                 0
room_given                  0
n_changes                   0
deposit                     0
agent_id                14299
company_id              97177
waitlist_days               0
customer_class              0
avg_daily_rate              0
parking_spaces              0
special_requests            0
dtype: int64

In [21]:

df = df.drop(columns=['company_id', 'agent_id'])
df = df.dropna()


X = df.drop(columns=['cancelled'])
y = df['cancelled']


X = pd.get_dummies(X, drop_first=True)

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    eval_metric='logloss',  
    random_state=42
)


In [23]:


xgb_model.fit(x_train, y_train)
y_pred = xgb_model.predict(x_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8496477387097822
              precision    recall  f1-score   support

           0       0.85      0.92      0.89     19456
           1       0.85      0.72      0.78     11345

    accuracy                           0.85     30801
   macro avg       0.85      0.82      0.83     30801
weighted avg       0.85      0.85      0.85     30801

